In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import re

# Инициализация Spark с 2 воркерами
spark = SparkSession.builder \
    .appName("NASALogAnalysis") \
    .master("local[2]") \
    .getOrCreate()

# Читаем лог
raw_df = spark.read.text("access.log")

# Регулярка для парсинга NASA лога
# Группы: 1-хост, 2-дата, 3-метод, 4-URL, 5-протокол, 6-статус, 7-размер
pattern = r'^(\S+) - - \[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2} -\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\d+|-)'

logs_df = raw_df.select(
    F.regexp_extract('value', pattern, 4).alias('url'),
    F.regexp_extract('value', pattern, 6).cast("int").alias('status'),
    F.to_date(F.substring(F.regexp_extract('value', pattern, 2), 1, 11), "dd/MMM/yyyy").alias('date')
).filter(F.col("url").rlike(r"(?i)\.html?$")) # Фильтруем только .html и .htm

# 1. ТОП-10 посещаемых URL
top_10_urls_df = logs_df.groupBy("url").count().orderBy(F.desc("count")).limit(10)
top_10_list = [row['url'] for row in top_10_urls_df.select("url").collect()]

print("Топ 10 URL:")
top_10_urls_df.show()

Топ 10 URL:
+--------------------+-----+
|                 url|count|
+--------------------+-----+
|           /ksc.html|43615|
|/shuttle/missions...|24580|
|/shuttle/missions...|22425|
|/software/winvn/w...|10337|
|/history/history....|10110|
|/history/apollo/a...| 8973|
|/shuttle/countdow...| 7856|
|/history/apollo/a...| 7158|
|/shuttle/technolo...| 6504|
|/shuttle/missions...| 5260|
+--------------------+-----+



In [2]:
# 2. Анализ кодов ответа для ТОП-10

# 1. Считаем ОБЩЕЕ кол-во запросов для каждого ТОП-10 URL (включая все статусы)
total_counts = logs_df.filter(F.col("url").isin(top_10_list)) \
    .groupBy("url") \
    .agg(F.count("*").alias("total_requests"))

# 2. Считаем кол-во конкретно по интересующим нас статусам
status_counts = logs_df.filter(F.col("url").isin(top_10_list)) \
    .groupBy("url", "status") \
    .count()

# 3. Соединяем и считаем долю от ОБЩЕГО числа
final_stats = status_counts.join(total_counts, "url") \
    .withColumn("share_%", F.round((F.col("count") / F.col("total_requests")) * 100, 2)) \
    .filter(F.col("status").isin([200, 404, 500])) # Оставляем только нужные по заданию

final_stats.orderBy(F.desc("count")).show()

+--------------------+------+-----+--------------+-------+
|                 url|status|count|total_requests|share_%|
+--------------------+------+-----+--------------+-------+
|           /ksc.html|   200|41324|         43615|  94.75|
|/shuttle/missions...|   200|23222|         24580|  94.48|
|/shuttle/missions...|   200|20689|         22425|  92.26|
|/history/history....|   200| 9481|         10110|  93.78|
|/software/winvn/w...|   200| 9243|         10337|  89.42|
|/history/apollo/a...|   200| 8313|          8973|  92.64|
|/history/apollo/a...|   200| 6661|          7158|  93.06|
|/shuttle/countdow...|   200| 6588|          7856|  83.86|
|/shuttle/technolo...|   200| 6244|          6504|   96.0|
|/shuttle/missions...|   200| 5098|          5260|  96.92|
+--------------------+------+-----+--------------+-------+



In [3]:
# 1. Группируем запросы по URL и по ДНЯМ
daily_traffic = logs_df.filter(F.col("url").isin(top_10_list)) \
    .groupBy("url", "date") \
    .count()

# 2. Считаем среднее (μ) и стандартное отклонение (σ) для каждого URL
stats = daily_traffic.groupBy("url").agg(
    F.mean("count").alias("avg"),
    F.stddev("count").alias("std")
)

# 3. Соединяем данные и ищем отклонения > 2σ
# Формула: |count - avg| > 2 * std
anomalies = daily_traffic.join(stats, "url") \
    .withColumn("deviation", F.abs(F.col("count") - F.col("avg"))) \
    .withColumn("is_anomaly", F.col("deviation") > (F.col("std") * 2)) \
    .withColumn("type", F.when(F.col("count") > F.col("avg"), "HIGH").otherwise("LOW"))

# Выводим только аномальные дни
print("Аномальные всплески и падения трафика:")
anomalies.filter("is_anomaly == true") \
    .select("url", "date", "count", "avg", "type") \
    .orderBy("url", "date").show()

Аномальные всплески и падения трафика:
+--------------------+----------+-----+------------------+----+
|                 url|      date|count|               avg|type|
+--------------------+----------+-----+------------------+----+
|/history/apollo/a...|1995-08-07|  373|             238.6|HIGH|
|/history/apollo/a...|1995-08-01|  182|             299.1| LOW|
|/history/apollo/a...|1995-08-26|  185|             299.1| LOW|
|/history/history....|1995-08-01|  173|             337.0| LOW|
|/shuttle/countdow...|1995-08-30|  973| 261.8666666666667|HIGH|
|/shuttle/countdow...|1995-08-31| 1660| 261.8666666666667|HIGH|
|/shuttle/missions...|1995-08-01|    9|175.33333333333334| LOW|
|/shuttle/missions...|1995-08-30|  322|175.33333333333334|HIGH|
|/shuttle/missions...|1995-08-31|  438|175.33333333333334|HIGH|
|/software/winvn/w...|1995-08-01|  155|344.56666666666666| LOW|
|/software/winvn/w...|1995-08-25|  539|344.56666666666666|HIGH|
+--------------------+----------+-----+------------------+----+

